# Predict 2026 World Cup Quarter-Finals

This notebook trains the selected model (recency- and importance-weighted L2 logistic
regression, as audited in notebooks `13` and `16`) on all matches before the
quarter-final round, then predicts the four 2026 FIFA World Cup quarter-finals:

| Date | Match | Venue |
| --- | --- | --- |
| 2026-07-09 | France vs Morocco | Foxborough (Boston) |
| 2026-07-10 | Spain vs Belgium | Inglewood (Los Angeles) |
| 2026-07-11 | Norway vs England | Miami Gardens (Miami) |
| 2026-07-11 | Argentina vs Switzerland | Kansas City |

The raw snapshot was refreshed from `martj42/international_results` so the processed
tables include all round-of-16 results through 2026-07-07. Training is frozen before
2026-07-09, so every prediction uses only information available before the first
quarter-final kicked off. The prediction uses only local repository data — no betting
odds, squads, injuries, or market values. Fixture home/away designations, dates, and
venues match the upstream scheduled rows.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
REPORTS = ROOT / 'reports'
FIRST_QF_DATE = pd.Timestamp('2026-07-09')
SEED = 20260707
LABELS = {'H': 0, 'D': 1, 'A': 2}
ORDERED = np.array(['H', 'D', 'A'])

historical = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
fixtures = pd.DataFrame([
    {'match_id': 'P20260709_FRA_MAR', 'source_row_number': -1, 'date': pd.Timestamp('2026-07-09'), 'home_team': 'France', 'away_team': 'Morocco', 'original_home_team': 'France', 'original_away_team': 'Morocco', 'home_score': 0, 'away_score': 0, 'tournament': 'FIFA World Cup', 'city': 'Foxborough', 'country': 'United States', 'neutral': True, 'result': 'H', 'goal_difference': 0},
    {'match_id': 'P20260710_ESP_BEL', 'source_row_number': -2, 'date': pd.Timestamp('2026-07-10'), 'home_team': 'Spain', 'away_team': 'Belgium', 'original_home_team': 'Spain', 'original_away_team': 'Belgium', 'home_score': 0, 'away_score': 0, 'tournament': 'FIFA World Cup', 'city': 'Inglewood', 'country': 'United States', 'neutral': True, 'result': 'H', 'goal_difference': 0},
    {'match_id': 'P20260711_NOR_ENG', 'source_row_number': -3, 'date': pd.Timestamp('2026-07-11'), 'home_team': 'Norway', 'away_team': 'England', 'original_home_team': 'Norway', 'original_away_team': 'England', 'home_score': 0, 'away_score': 0, 'tournament': 'FIFA World Cup', 'city': 'Miami Gardens', 'country': 'United States', 'neutral': True, 'result': 'H', 'goal_difference': 0},
    {'match_id': 'P20260711_ARG_SUI', 'source_row_number': -4, 'date': pd.Timestamp('2026-07-11'), 'home_team': 'Argentina', 'away_team': 'Switzerland', 'original_home_team': 'Argentina', 'original_away_team': 'Switzerland', 'home_score': 0, 'away_score': 0, 'tournament': 'FIFA World Cup', 'city': 'Kansas City', 'country': 'United States', 'neutral': True, 'result': 'H', 'goal_difference': 0},
])

for team in set(fixtures.home_team) | set(fixtures.away_team):
    assert ((historical.home_team == team) | (historical.away_team == team)).any(), f'missing history for {team}'
assert historical.date.max() == pd.Timestamp('2026-07-07'), 'processed data must include the round of 16'
wc2026 = historical[(historical.tournament == 'FIFA World Cup') & (historical.date.dt.year == 2026)]
assert len(wc2026) == 96, 'expected 96 completed 2026 World Cup matches'
historical.tail(4)[['date', 'home_team', 'away_team', 'tournament', 'home_score', 'away_score', 'result']]

,date,home_team,away_team,tournament,home_score,away_score,result
49497,2026-07-06,Portugal,Spain,FIFA World Cup,0,1,A
49498,2026-07-06,United States,Belgium,FIFA World Cup,1,4,A
49499,2026-07-07,Argentina,Egypt,FIFA World Cup,3,2,H
49500,2026-07-07,Switzerland,Colombia,FIFA World Cup,0,0,D


## Current Elo ratings for the fixtures

Fixture rows need the same pre-match Elo columns as historical rows. Ratings are
replayed chronologically over all completed matches (same configuration as
`02_elo_baseline.ipynb`: start 1500, K=20, +100 home advantage when not neutral),
then attached to the four quarter-final rows. No quarter-final result exists yet, so
ratings are identical for all four fixtures.

In [2]:
INITIAL_RATING = 1500.0
K_FACTOR = 20.0
DRAW_PRIOR = 0.25

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10.0 ** (-(rating_a - rating_b) / 400.0))

def actual_home_score(result):
    return {'H': 1.0, 'D': 0.5, 'A': 0.0}[result]

def outcome_probabilities(home_elo, away_elo, neutral):
    home_expectation = expected_score(home_elo, away_elo) if neutral else expected_score(home_elo + 100.0, away_elo)
    return ((1.0 - DRAW_PRIOR) * home_expectation, DRAW_PRIOR, (1.0 - DRAW_PRIOR) * (1.0 - home_expectation))

def current_ratings(frame):
    ordered = frame.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)
    ratings = {}
    for _, day in ordered.groupby('date', sort=True):
        start = ratings.copy()
        deltas = {}
        for row in day.itertuples(index=False):
            home_elo = start.get(row.home_team, INITIAL_RATING)
            away_elo = start.get(row.away_team, INITIAL_RATING)
            expected_home = expected_score(home_elo if bool(row.neutral) else home_elo + 100.0, away_elo)
            change = K_FACTOR * (actual_home_score(row.result) - expected_home)
            deltas[row.home_team] = deltas.get(row.home_team, 0.0) + change
            deltas[row.away_team] = deltas.get(row.away_team, 0.0) - change
        for team, delta in deltas.items():
            ratings[team] = start.get(team, INITIAL_RATING) + delta
    return ratings

ratings = current_ratings(historical)
fixture_elo_rows = []
for row in fixtures.itertuples(index=False):
    home_elo = ratings.get(row.home_team, INITIAL_RATING)
    away_elo = ratings.get(row.away_team, INITIAL_RATING)
    p_home, p_draw, p_away = outcome_probabilities(home_elo, away_elo, row.neutral)
    fixture_elo_rows.append({
        'home_elo': home_elo, 'away_elo': away_elo, 'elo_difference': home_elo - away_elo,
        'elo_adjusted_difference': home_elo - away_elo,
        'elo_home_probability': p_home, 'elo_draw_probability': p_draw, 'elo_away_probability': p_away,
    })
fixtures_with_elo = pd.concat([fixtures.reset_index(drop=True), pd.DataFrame(fixture_elo_rows)], axis=1)
combined = pd.concat([historical, fixtures_with_elo], ignore_index=True).sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)
fixtures_with_elo[['date', 'home_team', 'away_team', 'home_elo', 'away_elo', 'elo_difference', 'elo_home_probability', 'elo_draw_probability', 'elo_away_probability']]

,date,home_team,away_team,home_elo,away_elo,elo_difference,elo_home_probability,elo_draw_probability,elo_away_probability
0,2026-07-09,France,Morocco,1957.934677,1849.448882,108.485795,0.488430,0.25,0.261570
1,2026-07-10,Spain,Belgium,1995.051625,1867.353040,127.698585,0.506941,0.25,0.243059
2,2026-07-11,Norway,England,1784.984942,1902.073412,-117.088469,0.253199,0.25,0.496801
3,2026-07-11,Argentina,Switzerland,2026.694974,1817.279863,209.415111,0.577125,0.25,0.172875


## Feature pipeline

Verbatim copy of the feature pipeline from `13_world_cup_2026_results_vs_predictions.ipynb`.
All rolling features use `shift(1)`, so the fixture rows only see matches played before
them — the placeholder 0-0 scores never leak into their own features.

In [3]:
def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    markers = ['copa america', 'copa américa', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']
    if any(marker in text for marker in markers):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame['wc_match_number'] = -1
    mask = frame.tournament.eq('FIFA World Cup')
    frame.loc[mask, 'wc_match_number'] = frame.loc[mask].groupby(frame.loc[mask, 'date'].dt.year).cumcount()
    frame['wc_group_stage'] = frame.wc_match_number.between(0, 47).astype(float)
    frame['wc_knockout_stage'] = (frame.wc_match_number >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {'H': 1.0, 'D': 0.5, 'A': 0.0}
    home = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.home_team, 'is_home': True,
        'goals_for': frame.home_score.astype(float), 'goals_against': frame.away_score.astype(float),
        'team_elo': frame.home_elo.astype(float), 'result_points': frame.result.map(points).astype(float),
        'expected_points': frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.away_team, 'is_home': False,
        'goals_for': frame.away_score.astype(float), 'goals_against': frame.home_score.astype(float),
        'team_elo': frame.away_elo.astype(float), 'result_points': 1.0 - frame.result.map(points).astype(float),
        'expected_points': frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = pd.concat([home, away], ignore_index=True).sort_values(['team', 'date', 'match_id'], kind='mergesort').reset_index(drop=True)
    grouped = panel.groupby('team', sort=False)
    panel['days_since_match'] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    recent_counts = pd.Series(0.0, index=panel.index)
    for _, idx in panel.groupby('team', sort=False).groups.items():
        dates = panel.loc[idx, 'date'].to_numpy()
        counts = []
        for pos, current in enumerate(dates):
            prior = dates[:pos]
            counts.append(float(np.sum((current - prior) <= np.timedelta64(365, 'D'))))
        recent_counts.loc[idx] = counts
    panel['recent_matches_365'] = recent_counts
    for window in [3, 5, 10]:
        for col in ['goals_for', 'goals_against', 'result_points']:
            fill = {'goals_for': 1.25, 'goals_against': 1.25, 'result_points': 0.5}[col]
            panel[f'rolling_{window}_{col}'] = grouped[col].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()).fillna(fill)
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f'rolling_{window}_form_vs_expected'] = (actual - expected).fillna(0.0)
        panel[f'rolling_{window}_elo_trend'] = (grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)).fillna(0.0)
    return panel

def build_features(frame):
    frame = add_world_cup_stage(frame)
    frame['tournament_type'] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith('rolling_')] + ['days_since_match', 'recent_matches_365']
    home = panel.loc[panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'home_{c}' for c in value_cols})
    away = panel.loc[~panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'away_{c}' for c in value_cols})
    out = frame.merge(home, on='match_id', how='left').merge(away, on='match_id', how='left')
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    for window in [3, 5, 10]:
        out[f'recent_goal_total_{window}'] = out[f'home_rolling_{window}_goals_for'] + out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_total_{window}'] = out[f'home_rolling_{window}_goals_against'] + out[f'away_rolling_{window}_goals_against']
        out[f'recent_form_gap_{window}'] = out[f'home_rolling_{window}_result_points'] - out[f'away_rolling_{window}_result_points']
        out[f'recent_attack_gap_{window}'] = out[f'home_rolling_{window}_goals_for'] - out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_gap_{window}'] = out[f'home_rolling_{window}_goals_against'] - out[f'away_rolling_{window}_goals_against']
    out['rest_gap'] = out.home_days_since_match - out.away_days_since_match
    out['activity_gap_365'] = out.home_recent_matches_365 - out.away_recent_matches_365
    return out.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)

featured = build_features(combined)
assert featured.filter(regex='rolling_|days_since|recent_|elo_similarity|stage|gap').isna().sum().sum() == 0
featured.shape

(49505, 80)

## Train the frozen model and predict

Same design matrix, sample weights, and hyperparameters as the live audit in
notebook `13`: weighted L2 logistic regression, `C=0.25`, recency half-life of
7 years, and tournament-importance weights. Training uses every match from
2000-01-01 up to 2026-07-07 inclusive — nothing on or after the first
quarter-final date.

In [4]:
FEATURES = [
    'elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral', 'home_elo', 'away_elo', 'wc_group_stage', 'wc_knockout_stage',
    'home_days_since_match', 'away_days_since_match', 'home_recent_matches_365', 'away_recent_matches_365', 'rest_gap', 'activity_gap_365',
]
for window in [3, 5, 10]:
    FEATURES += [
        f'home_rolling_{window}_goals_for', f'home_rolling_{window}_goals_against', f'home_rolling_{window}_result_points', f'home_rolling_{window}_form_vs_expected', f'home_rolling_{window}_elo_trend',
        f'away_rolling_{window}_goals_for', f'away_rolling_{window}_goals_against', f'away_rolling_{window}_result_points', f'away_rolling_{window}_form_vs_expected', f'away_rolling_{window}_elo_trend',
        f'recent_goal_total_{window}', f'recent_defence_total_{window}', f'recent_form_gap_{window}', f'recent_attack_gap_{window}', f'recent_defence_gap_{window}',
    ]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def design(frame):
    out = frame[FEATURES].astype(float).copy()
    for col in [c for c in out.columns if 'elo' in c or c in ['home_elo', 'away_elo', 'elo_difference', 'abs_elo_difference']]:
        out[col] = out[col] / 400.0
    for col in ['home_days_since_match', 'away_days_since_match', 'rest_gap']:
        out[col] = out[col] / 30.0
    out['neutral'] = frame.neutral.astype(float)
    for level in TOURNAMENT_LEVELS:
        out[f'tournament_{level}'] = (frame.tournament_type == level).astype(float)
    return out

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 7.0)
    type_weight = frame.tournament_type.map({'world_cup': 1.7, 'qualifier': 1.25, 'continental': 1.15, 'friendly': 0.65, 'other': 1.0}).fillna(1.0)
    return np.asarray(recency * type_weight, dtype=float)

def normalize(probs):
    probs = np.clip(np.asarray(probs, dtype=float), 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

train = featured[(featured.date < FIRST_QF_DATE) & (featured.date >= '2000-01-01')].copy()
today = featured[featured.match_id.str.startswith('P2026')].copy().sort_values(['date', 'match_id']).reset_index(drop=True)
assert train.date.max() == pd.Timestamp('2026-07-07') and len(today) == 4

model = make_pipeline(StandardScaler(), LogisticRegression(C=0.25, max_iter=3000, random_state=SEED))
model.fit(design(train), train.result.map(LABELS).to_numpy(), logisticregression__sample_weight=sample_weights(train))
probs = normalize(model.predict_proba(design(today)))

predictions = today[['date', 'home_team', 'away_team', 'city', 'country', 'home_elo', 'away_elo', 'elo_difference']].copy()
predictions['home_win_probability'] = probs[:, 0]
predictions['draw_probability'] = probs[:, 1]
predictions['away_win_probability'] = probs[:, 2]
predictions['predicted_1x2'] = ORDERED[probs.argmax(axis=1)]
predictions['predicted_result'] = np.select(
    [predictions.predicted_1x2.eq('H'), predictions.predicted_1x2.eq('D')],
    [predictions.home_team + ' win', 'draw'],
    default=predictions.away_team + ' win',
)
predictions[['date', 'home_team', 'away_team', 'home_win_probability', 'draw_probability', 'away_win_probability', 'predicted_result']]

,date,home_team,away_team,home_win_probability,draw_probability,away_win_probability,predicted_result
0,2026-07-09,France,Morocco,0.552414,0.230319,0.217267,France win
1,2026-07-10,Spain,Belgium,0.544191,0.249659,0.206151,Spain win
2,2026-07-11,Argentina,Switzerland,0.670585,0.218071,0.111344,Argentina win
3,2026-07-11,Norway,England,0.289645,0.232630,0.477726,England win


In [5]:
REPORTS.mkdir(exist_ok=True)
predictions.to_csv(REPORTS / 'world_cup_2026_quarter_final_predictions.csv', index=False)

lines = [
    '| Date | Match | Model pick | P(home) | P(draw) | P(away) | Actual | Result |',
    '| --- | --- | --- | --- | --- | --- | --- | :---: |',
]
for row in predictions.itertuples(index=False):
    pick = row.home_team if row.predicted_1x2 == 'H' else 'Draw' if row.predicted_1x2 == 'D' else row.away_team
    lines.append(
        f'| {row.date.date().isoformat()} | {row.home_team} vs {row.away_team} | {pick} '
        f'| {row.home_win_probability:.0%} | {row.draw_probability:.0%} | {row.away_win_probability:.0%} | — | ⏳ |'
    )
table_text = '\n'.join(lines)
(REPORTS / 'world_cup_2026_quarter_final_readme_table.txt').write_text(table_text + '\n', encoding='utf-8')
print(table_text)

| Date | Match | Model pick | P(home) | P(draw) | P(away) | Actual | Result |
| --- | --- | --- | --- | --- | --- | --- | :---: |
| 2026-07-09 | France vs Morocco | France | 55% | 23% | 22% | — | ⏳ |
| 2026-07-10 | Spain vs Belgium | Spain | 54% | 25% | 21% | — | ⏳ |
| 2026-07-11 | Argentina vs Switzerland | Argentina | 67% | 22% | 11% | — | ⏳ |
| 2026-07-11 | Norway vs England | England | 29% | 23% | 48% | — | ⏳ |


## Results pending

The four quarter-finals had not finished when this notebook was frozen. Once final
scores land in the upstream dataset, refresh `data/raw/international_results.csv`,
rebuild the processed tables, and rerun the audit in notebook `13` (extended to the
new dates) to score these picks.